In [0]:
-- ============================================================
-- AEGIS — ANALYTICAL TOOL LAYER
-- Agent-ready Unity Catalog functions
-- ============================================================


-- ============================================================
-- 1. SUPPLIER PERFORMANCE
-- ============================================================

CREATE OR REPLACE FUNCTION
workspace.aegis_gold.get_supplier_performance(
    p_supplier_id STRING
        COMMENT 'Supplier identifier to analyze, for example SUP0014.',
    p_start_date DATE
        COMMENT 'Start date of the analysis period. Results are returned at monthly grain.',
    p_end_date DATE
        COMMENT 'End date of the analysis period. Results are returned through the month containing this date.'
)
RETURNS TABLE
COMMENT 'Returns monthly supplier performance metrics for a specific supplier, including shipment volume, revenue, transit time, delivery delay, late delivery rate, product coverage, supplier reliability, and agreed lead time. Use this tool when investigating supplier performance, deterioration, or delivery concerns.'
RETURN
SELECT
    month,
    supplier_id,
    supplier_name,
    region,
    supplier_type,
    reliability_score,
    lead_time_variability,
    orders,
    units_supplied,
    revenue,
    shipments,
    avg_actual_transit_days,
    avg_delivery_delay_days,
    late_delivery_rate,
    product_count,
    avg_agreed_lead_time
FROM workspace.aegis_gold.supplier_performance
WHERE supplier_id = p_supplier_id
  AND month >= DATE_TRUNC('month', p_start_date)
  AND month <= DATE_TRUNC('month', p_end_date)
ORDER BY month;


-- ============================================================
-- 2. DELIVERY PERFORMANCE
-- ============================================================

CREATE OR REPLACE FUNCTION
workspace.aegis_gold.get_delivery_performance(
    p_warehouse_id STRING
        COMMENT 'Optional warehouse identifier, for example WH0003. Use NULL when the investigation is not restricted to a specific warehouse.',
    p_carrier_id STRING
        COMMENT 'Optional carrier identifier, for example CAR0007. Use NULL when the investigation is not restricted to a specific carrier.',
    p_start_date DATE
        COMMENT 'Start date of the analysis period.',
    p_end_date DATE
        COMMENT 'End date of the analysis period.'
)
RETURNS TABLE
COMMENT 'Returns monthly delivery performance for a specified warehouse, carrier, or both. Includes shipment volume, late shipments, late delivery rate, transit times, delivery delays, and maximum delay. Use this tool to investigate warehouse or carrier delivery performance.'
RETURN
SELECT
    month,
    warehouse_id,
    warehouse_region,
    carrier_id,
    carrier_name,
    service_type,
    shipment_count,
    late_shipments,
    late_delivery_rate,
    avg_actual_transit_days,
    avg_expected_transit_days,
    avg_delivery_delay_days,
    max_delivery_delay_days
FROM workspace.aegis_gold.delivery_performance
WHERE (p_warehouse_id IS NULL OR warehouse_id = p_warehouse_id)
  AND (p_carrier_id IS NULL OR carrier_id = p_carrier_id)
  AND month >= DATE_TRUNC('month', p_start_date)
  AND month <= DATE_TRUNC('month', p_end_date)
ORDER BY month;


-- ============================================================
-- 3. INVENTORY HEALTH
-- ============================================================

CREATE OR REPLACE FUNCTION
workspace.aegis_gold.get_inventory_health(
    p_warehouse_id STRING
        COMMENT 'Optional warehouse identifier, for example WH0003. Use NULL to analyze inventory across warehouses.',
    p_product_id STRING
        COMMENT 'Optional product identifier, for example PROD000123. Use NULL to analyze inventory across products.',
    p_start_date DATE
        COMMENT 'Start date of the inventory analysis period.',
    p_end_date DATE
        COMMENT 'End date of the inventory analysis period.'
)
RETURNS TABLE
COMMENT 'Returns daily inventory health for a specified warehouse, product, or both. Includes stock, reserved quantity, available inventory, stockout status, demand index, and inventory-to-demand ratio. Use this tool to investigate stockouts, inventory shortages, and inventory anomalies.'
RETURN
SELECT
    snapshot_date,
    warehouse_id,
    product_id,
    product_name,
    category,
    warehouse_region,
    warehouse_capacity,
    stock_quantity,
    reserved_quantity,
    available_quantity,
    stockout_flag,
    stockout_indicator,
    demand_index,
    inventory_demand_ratio
FROM workspace.aegis_gold.inventory_health
WHERE (p_warehouse_id IS NULL OR warehouse_id = p_warehouse_id)
  AND (p_product_id IS NULL OR product_id = p_product_id)
  AND snapshot_date >= p_start_date
  AND snapshot_date <= p_end_date
ORDER BY snapshot_date;


-- ============================================================
-- 4. WAREHOUSE PERFORMANCE
-- ============================================================

CREATE OR REPLACE FUNCTION
workspace.aegis_gold.get_warehouse_performance(
    p_warehouse_id STRING
        COMMENT 'Warehouse identifier to analyze, for example WH0003.',
    p_start_date DATE
        COMMENT 'Start date of the analysis period.',
    p_end_date DATE
        COMMENT 'End date of the analysis period.'
)
RETURNS TABLE
COMMENT 'Returns monthly operational performance for a specific warehouse, including shipment volume, late shipments, late delivery rate, transit and delivery delays, inventory levels, stockout rate, warehouse capacity, and processing efficiency. Use this tool to investigate possible warehouse bottlenecks or operational deterioration.'
RETURN
SELECT
    month,
    warehouse_id,
    region,
    warehouse_type,
    capacity,
    processing_efficiency,
    shipments,
    late_shipments,
    late_delivery_rate,
    avg_actual_transit_days,
    avg_delivery_delay_days,
    avg_stock_quantity,
    avg_available_quantity,
    stockout_rate
FROM workspace.aegis_gold.warehouse_performance
WHERE warehouse_id = p_warehouse_id
  AND month >= DATE_TRUNC('month', p_start_date)
  AND month <= DATE_TRUNC('month', p_end_date)
ORDER BY month;


-- ============================================================
-- 5. CARRIER PERFORMANCE
-- ============================================================

CREATE OR REPLACE FUNCTION
workspace.aegis_gold.get_carrier_performance(
    p_carrier_id STRING
        COMMENT 'Carrier identifier to analyze, for example CAR0007.',
    p_start_date DATE
        COMMENT 'Start date of the analysis period.',
    p_end_date DATE
        COMMENT 'End date of the analysis period.'
)
RETURNS TABLE
COMMENT 'Returns monthly delivery performance for a specific carrier, including shipment volume, late shipments, late delivery rate, actual and expected transit time, delivery delay, and transit-time variability. Use this tool to investigate carrier deterioration or delivery reliability concerns.'
RETURN
SELECT
    month,
    carrier_id,
    carrier_name,
    service_type,
    base_transit_days,
    transit_variability,
    shipment_count,
    late_shipments,
    late_delivery_rate,
    avg_actual_transit_days,
    avg_delivery_delay_days,
    transit_time_stddev
FROM workspace.aegis_gold.carrier_performance
WHERE carrier_id = p_carrier_id
  AND month >= DATE_TRUNC('month', p_start_date)
  AND month <= DATE_TRUNC('month', p_end_date)
ORDER BY month;


-- ============================================================
-- 6. SUPPLY CHAIN RISK
-- ============================================================

CREATE OR REPLACE FUNCTION
workspace.aegis_gold.get_supply_chain_risk(
    p_supplier_id STRING
        COMMENT 'Supplier identifier to analyze, for example SUP0014.',
    p_start_date DATE
        COMMENT 'Start date of the analysis period.',
    p_end_date DATE
        COMMENT 'End date of the analysis period.'
)
RETURNS TABLE
COMMENT 'Returns monthly deterministic supply-chain risk signals for a specific supplier, including delivery, delay, inventory, supplier reliability signals, and the resulting risk signal score. The score is a transparent screening signal, not a probability and does not use ground-truth incident information. Use this tool to identify suppliers requiring further investigation.'
RETURN
SELECT
    month,
    supplier_id,
    reliability_score,
    lead_time_variability,
    late_delivery_rate,
    avg_delivery_delay_days,
    avg_actual_transit_days,
    avg_agreed_lead_time,
    stockout_rate,
    delivery_risk_signal,
    delay_risk_signal,
    inventory_risk_signal,
    supplier_reliability_signal,
    risk_signal_score
FROM workspace.aegis_gold.supply_chain_risk
WHERE supplier_id = p_supplier_id
  AND month >= DATE_TRUNC('month', p_start_date)
  AND month <= DATE_TRUNC('month', p_end_date)
ORDER BY month;

CREATE OR REPLACE FUNCTION workspace.aegis_gold.get_supplier_network(
    p_supplier_id STRING,
    p_start_date DATE,
    p_end_date DATE
)
RETURNS TABLE (
    supplier_id STRING,
    supplier_name STRING,
    warehouse_id STRING,
    carrier_id STRING,
    shipment_count BIGINT,
    units_shipped BIGINT
)
LANGUAGE SQL
COMMENT 'Returns the operational network connecting a supplier to warehouses and carriers during a specified period.'
PARAMETER STYLE SQL
READS SQL DATA
RETURN
    SELECT
        sp.supplier_id,
        s.supplier_name,
        sh.warehouse_id,
        sh.carrier_id,
        COUNT(DISTINCT sh.shipment_id) AS shipment_count,
        SUM(si.quantity) AS units_shipped
    FROM workspace.aegis_silver.silver_supplier_product sp
    JOIN workspace.aegis_silver.silver_supplier s
        ON sp.supplier_id = s.supplier_id
    JOIN workspace.aegis_silver.silver_order_item oi
        ON sp.supplier_id = oi.supplier_id
        AND sp.product_id = oi.product_id
    JOIN workspace.aegis_silver.silver_shipment_item si
        ON oi.order_item_id = si.order_item_id
    JOIN workspace.aegis_silver.silver_shipment sh
        ON si.shipment_id = sh.shipment_id
    WHERE sp.supplier_id = p_supplier_id
      AND CAST(sh.shipment_date AS DATE)
            BETWEEN p_start_date AND p_end_date
    GROUP BY
        sp.supplier_id,
        s.supplier_name,
        sh.warehouse_id,
        sh.carrier_id
    ORDER BY
        shipment_count DESC;